In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score

In [ ]:
df = pd.read_csv('smote_balanced_dataset.csv')
df.head()

In [ ]:
X = df.drop('CLASS', axis=1)
y = df['CLASS']

In [ ]:
numeric_cols = ['AGE', 'WARD_encoded', 'MONTH_sin', 'MONTH_cos']
categorical_cols = [col for col in X.columns if col not in numeric_cols]

In [ ]:
scaler = StandardScaler()
X_scaled = X.copy()  # Preserve original X
X_scaled[numeric_cols] = scaler.fit_transform(X[numeric_cols])  # Scale only numeric columns

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
print("Training set shape:", X_train.shape, y_train.shape)
print("Test set shape:", X_test.shape, y_test.shape)

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
xgb = XGBClassifier(random_state=42, eval_metric='logloss')
param_grid = {
    'n_estimators': [50, 100, 200],      # Number of trees
    'max_depth': [3, 6, 9],              # Maximum depth of trees
    'learning_rate': [0.01, 0.1, 0.3],   # Step size shrinkage
    'subsample': [0.8, 1.0]              # Fraction of samples used per tree
}

In [ ]:
grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    cv=skf,
    scoring='f1',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

In [ ]:
print("Best parameters from GridSearchCV:", grid_search.best_params_)
print("Best cross-validation F1-score:", grid_search.best_score_)

In [ ]:
best_model = grid_search.best_estimator_
best_model.fit(X_train, y_train)

In [ ]:
y_pred = best_model.predict(X_test)

In [ ]:
round(accuracy_score(y_test, y_pred)*100,2)

In [ ]:
print(classification_report(y_test, y_pred, target_names=['Dengue (0)', 'Malaria (1)']))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Dengue (0)', 'Malaria (1)'], yticklabels=['Dengue (0)', 'Malaria (1)'])
plt.title('Confusion Matrix for XGBoost')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('xgboost_cm_bfrREsmpl.png', dpi=400)
plt.show()

# **LIME Explanation**

In [ ]:
!pip install lime

In [ ]:
import lime
import lime.lime_tabular
from lime.lime_tabular import LimeTabularExplainer

In [ ]:
lime_explainer = LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=X.columns,
    class_names=['Dengue (0)', 'Malaria (1)'],
    mode='classification',
    random_state=42
)

In [ ]:
instance_idx = 0
instance = X_test.iloc[instance_idx].values
lime_exp = lime_explainer.explain_instance(
    data_row=instance,
    predict_fn=best_model.predict_proba,
    num_features=len(X.columns)  # Include all features to aggregate later
)

In [ ]:
len(X.columns)

In [ ]:
instance


In [ ]:
len(instance)

In [ ]:
num_instances = 10

print("The 10 instances that WILL be used:")
print(X_test.iloc[:num_instances])

In [ ]:
print("Dataset indices of the 10 instances:")
print(X_test.index[:num_instances].tolist())

In [ ]:
lime_exp.show_in_notebook()

In [ ]:
plt.figure()
lime_exp.as_pyplot_figure()
plt.title('LIME Explanation for First Instance (XGBoost)')
plt.tight_layout()
plt.savefig('lime_explanation_instance_0_logistic.png', dpi=400)
plt.show()

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

lime_explainer = LimeTabularExplainer(
    training_data = X_train.values,
    feature_names = X_train.columns.tolist(),
    class_names = ['Dengue(0)', 'Malaria(1)'],   # 0 → Dengue, 1 → Malaria
    mode = 'classification'
)

In [ ]:
num_instances = 10

for i in range(num_instances):
    instance = X_test.iloc[i]

    explanation = lime_explainer.explain_instance(
        instance,
        best_model.predict_proba,
        num_features=len(X.columns)  # show all  features
    )

    # Show explanation
    explanation.show_in_notebook()

In [ ]:
num_instances = 10

for i in range(num_instances):
    print(f"\nInstance {i}")
    print("Dataset index:", X_test.index[i])
    print(X_test.iloc[i])


In [ ]:
num_instances = 10
for i in range(num_instances):
    instance = X_test.iloc[i]
    #explanation = lime_explainer.explain_instance(instance, best_model.predict_proba, num_features=5)
    explanation = lime_explainer.explain_instance(instance, best_model.predict_proba, num_features=len(X.columns))
    a = explanation.as_list()
    # print(explanation.as_list())
    print(a)
    print("\n")

In [ ]:
all_explanations = []

num_instances = 10
for i in range(num_instances):
    instance = X_test.iloc[i]
    explanation = lime_explainer.explain_instance(
        instance,
        best_model.predict_proba,
        num_features=len(X.columns)  # ask for all 22
    )
    all_explanations.append(explanation.as_list())

In [ ]:
from collections import defaultdict
import numpy as np

feature_weights = defaultdict(list)

for exp in all_explanations:
    for feature_cond, weight in exp:
        for col in X.columns:
            if col in feature_cond:
                feature_weights[col].append(abs(weight))
                break

In [ ]:
avg_weight_df = pd.DataFrame({
    "Feature": X.columns,
    "Avg_Absolute_Weight": [
        np.mean(feature_weights[col]) if len(feature_weights[col]) > 0 else 0
        for col in X.columns
    ]
}).sort_values(by="Avg_Absolute_Weight", ascending=False)

print(avg_weight_df)


In [ ]:
len(avg_weight_df)


In [ ]:
thresholds = sorted(
    avg_weight_df["Avg_Absolute_Weight"].unique()
)

print("Threshold values:")
for t in thresholds:
    print(t)



In [ ]:
len(thresholds)

In [ ]:
# Fix ONE sample index
sample_index = 0
x_sample = X_test.iloc[sample_index]

comparison = []

for t in thresholds:
    selected_features = avg_weight_df[
        avg_weight_df["Avg_Absolute_Weight"] >= t
    ]["Feature"].tolist()

    comparison.append({
        "Threshold": t,
        "Num_Selected_Features": len(selected_features),
        "Selected_Features": selected_features
    })

comparison_df = pd.DataFrame(comparison)
print(comparison_df)


In [ ]:
all_samples_comparison = []

num_instances = 10

for i in range(num_instances):
    sample_row = {
        "Sample_Index": i,
        "Threshold_Comparison": []
    }

    for t in thresholds:
        selected_features = avg_weight_df[
            avg_weight_df["Avg_Absolute_Weight"] >= t
        ]["Feature"].tolist()

        sample_row["Threshold_Comparison"].append({
            "Threshold": t,
            "Num_Selected_Features": len(selected_features),
            "Selected_Features": selected_features
        })

    all_samples_comparison.append(sample_row)


In [ ]:
for sample in all_samples_comparison:
    print(f"\n================ Sample {sample['Sample_Index']} ================")

    for item in sample["Threshold_Comparison"]:
        print(f"Threshold: {item['Threshold']}")
        print(f"Num Selected Features: {item['Num_Selected_Features']}")
        print(f"Selected Features: {item['Selected_Features']}")
        print("-" * 60)

In [ ]:
rows = []

for sample in all_samples_comparison:
    for item in sample["Threshold_Comparison"]:
        rows.append({
            "Sample_Index": sample["Sample_Index"],
            "Threshold": item["Threshold"],
            "Num_Selected_Features": item["Num_Selected_Features"],
            "Selected_Features": item["Selected_Features"]
        })

comparison_df = pd.DataFrame(rows)
print(comparison_df)

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score


In [ ]:
results = []

In [ ]:
for _, row in comparison_df.iterrows():

    sample_index = row["Sample_Index"]
    threshold = row["Threshold"]
    selected_features = row["Selected_Features"]

    print("\n===================================")
    print(f"Sample {sample_index}")
    print(f"Threshold = {threshold}")
    print("Selected Features:", selected_features)

    # Safety check
    if len(selected_features) == 0:
        print("No features selected. Skipping.")
        continue

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------------------------
# Take unique thresholds & features (Sample 0)
# ---------------------------------------------
sample0_df = comparison_df[comparison_df["Sample_Index"] == 0]

final_results_xgb = []

for _, row in sample0_df.iterrows():

    threshold = row["Threshold"]
    selected_features = row["Selected_Features"]

    print("\n========================================")
    print(f"Threshold = {threshold}")
    print("Selected Features:", selected_features)
    print("Number of Features:", len(selected_features))

    # -------------------------------
    # 1. Create reduced datasets
    # -------------------------------
    X_train_selected = X_train[selected_features]
    X_test_selected  = X_test[selected_features]

    # -------------------------------
    # 2. StratifiedKFold + GridSearch (XGBoost)
    # -------------------------------
    skf_xgb = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    xgb_LIME = XGBClassifier(
        random_state=42,
        eval_metric='logloss',
        use_label_encoder=False
    )

    param_grid_LIME = {
        'n_estimators': [50, 100, 200],
        'max_depth': [3, 6, 9],
        'learning_rate': [0.01, 0.1, 0.3],
        'subsample': [0.8, 1.0]
    }

    grid_search_xgb = GridSearchCV(
        estimator=xgb_LIME,
        param_grid=param_grid_LIME,
        cv=skf_xgb,
        scoring='f1',
        n_jobs=-1
    )

    grid_search_xgb.fit(X_train_selected, y_train)

    print("Best XGB Parameters:", grid_search_xgb.best_params_)
    print("Best CV F1-score:", grid_search_xgb.best_score_)

    # -------------------------------
    # 3. Best XGB model
    # -------------------------------
    best_xgb_model = grid_search_xgb.best_estimator_
    best_xgb_model.fit(X_train_selected, y_train)

    # -------------------------------
    # 4. Test on FIRST 10 rows
    # -------------------------------
    X_test_subset = X_test_selected.iloc[:10]
    y_test_subset = y_test.iloc[:10]

    y_pred_subset = best_xgb_model.predict(X_test_subset)

    acc_subset = round(accuracy_score(y_test_subset, y_pred_subset) * 100, 2)
    print("Subset Accuracy (%):", acc_subset)

    print("\nClassification Report (Subset):")
    print(classification_report(
        y_test_subset,
        y_pred_subset,
        target_names=['Dengue (0)', 'Malaria (1)']
    ))

    # -------------------------------
    # 5. Confusion Matrix (BLUE-WHITE)
    # -------------------------------
    cm = confusion_matrix(y_test_subset, y_pred_subset)

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',   # 🔵 SAME AS EXISTING CODE
        xticklabels=['Dengue (0)', 'Malaria (1)'],
        yticklabels=['Dengue (0)', 'Malaria (1)']
    )
    plt.title(f'XGBoost Confusion Matrix (Threshold={round(threshold,4)})')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

    # -------------------------------
    # 6. Store results
    # -------------------------------
    final_results_xgb.append({
        "Threshold": threshold,
        "Num_Features": len(selected_features),
        "Subset_Accuracy": acc_subset,
        "CV_F1": grid_search_xgb.best_score_,
        "Selected_Features": selected_features
    })


## **SHAP Explanations**

In [ ]:
!pip install shap

In [ ]:
import shap

In [ ]:
explainer = shap.Explainer(best_model.predict, X_train)
shap_values = explainer(X_test)


In [ ]:
X_test_subset = X_test.iloc[:50]

In [ ]:
shap_values = explainer(X_test_subset)
shap_values

In [ ]:
feature_names = X_test_subset.columns.tolist()

In [ ]:
shap_vals_all_classes = shap_values.values
shap_vals_all_classes

In [ ]:
len(shap_values.values)

In [ ]:
len(shap_vals_all_classes)

In [ ]:
len(shap_vals_all_classes)

In [ ]:
shap.plots.beeswarm(shap_values)

In [ ]:
shap_vals_class_0 = shap_vals_all_classes[:,0]
shap_vals_class_0

In [ ]:
shap_vals_class_1 = shap_vals_all_classes[:, 1]
shap_vals_class_1

In [ ]:
shap_vals_class_0 = shap_vals_all_classes[:, 0]
shap_vals_class_1 = shap_vals_all_classes[:, 1]

In [ ]:
shap_vals_class_0

In [ ]:
shap_vals_class_1

In [ ]:
import numpy as np

mean_abs_shap = np.mean(np.abs(shap_values.values), axis=0)
sorted_idx = np.argsort(mean_abs_shap)[::-1]
sorted_features = np.array(X_test.columns)[sorted_idx]
sorted_values = mean_abs_shap[sorted_idx]

In [ ]:
sorted_values

In [ ]:
mean_abs_shap

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(sorted_features, sorted_values)
plt.xlabel("Mean |SHAP value|")
plt.title("Global Feature Importance Using SHAP")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
from xgboost import XGBClassifier


In [ ]:
from collections import defaultdict
import numpy as np

shap_thresholds = sorted(np.unique(mean_abs_shap))

shap_feature_selection = []

for t in shap_thresholds:
    selected_features_shap = sorted_features[sorted_values >= t].tolist()
    shap_feature_selection.append({
        "Threshold": t,
        "Num_Selected_Features": len(selected_features_shap),
        "Selected_Features": selected_features_shap
    })

shap_feature_selection_df = pd.DataFrame(shap_feature_selection)

shap_feature_selection_df_sorted = shap_feature_selection_df.sort_values(
    by="Num_Selected_Features",
    ascending=False
).reset_index(drop=True)

In [ ]:
final_results_shap_xgb = []

# Sort from highest number of features to lowest (22 → 1)
shap_feature_selection_df_sorted = shap_feature_selection_df.sort_values(
    by="Num_Selected_Features",
    ascending=False
).reset_index(drop=True)

for _, row in shap_feature_selection_df_sorted.iterrows():

    threshold = row["Threshold"]
    selected_features = row["Selected_Features"]

    print("\n===================================")
    print("Number of Features:", len(selected_features))
    print("Threshold:", threshold)

    # ----------------------------
    # 1. Reduce datasets
    # ----------------------------
    X_train_selected = X_train[selected_features]
    X_test_selected  = X_test[selected_features]

    # ----------------------------
    # 2. GridSearch + StratifiedKFold
    # ----------------------------
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    xgb = XGBClassifier(
        random_state=42,
        eval_metric='logloss',
        use_label_encoder=False
    )

    param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [3, 6, 9],
        'learning_rate': [0.01, 0.1, 0.3],
        'subsample': [0.8, 1.0]
    }

    grid = GridSearchCV(
        estimator=xgb,
        param_grid=param_grid,
        cv=skf,
        scoring='f1',
        n_jobs=-1
    )

    grid.fit(X_train_selected, y_train)

    best_model = grid.best_estimator_

    # ----------------------------
    # 3. Test on first 50 samples
    # ----------------------------
    X_test_subset = X_test_selected.iloc[:50]
    y_test_subset = y_test.iloc[:50]

    y_pred = best_model.predict(X_test_subset)

    acc = accuracy_score(y_test_subset, y_pred)
    f1  = f1_score(y_test_subset, y_pred)

    print("Accuracy:", round(acc*100, 2))
    print("F1 Score:", round(f1, 4))

    print("\nClassification Report:")
    print(classification_report(
        y_test_subset,
        y_pred,
        target_names=['Dengue (0)', 'Malaria (1)']
    ))

    # ----------------------------
    # 4. Confusion Matrix (BLUE)
    # ----------------------------
    cm = confusion_matrix(y_test_subset, y_pred)

    plt.figure(figsize=(6,5))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=['Dengue (0)', 'Malaria (1)'],
        yticklabels=['Dengue (0)', 'Malaria (1)']
    )
    plt.title(f'XGBoost SHAP Confusion Matrix ({len(selected_features)} Features)')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

    # ----------------------------
    # 5. Store results
    # ----------------------------
    final_results_shap_xgb.append({
        "Num_Features": len(selected_features),
        "Threshold": threshold,
        "Accuracy": acc,
        "F1_Score": f1
    })
